In [21]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Current working directory:")
print(PROJECT_ROOT)

print("\nFiles/folders here:")
print(sorted(p.name for p in PROJECT_ROOT.iterdir()))


Current working directory:
C:\Users\katri\Desktop\sc4000git\SC4000_ML

Files/folders here:
['.git', '.gitignore', 'README.md', 'data', 'notebooks', 'requirements.txt', 'submissions']


In [22]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

print((DATA_DIR / "men_processed.csv").exists())
print((DATA_DIR / "women_processed.csv").exists())


True
True


In [23]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss


In [24]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MEN_PATH = DATA_DIR / "men_processed.csv"
WOMEN_PATH = DATA_DIR / "women_processed.csv"

men = pd.read_csv(MEN_PATH)
women = pd.read_csv(WOMEN_PATH)

print("Men:", men.shape)
print("Women:", women.shape)

display(men.head())
display(women.head())


Men: (1449, 8)
Women: (961, 8)


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target
0,2003,1411,1421,0,9.208046,0.151724,0.045943,0
1,2003,1112,1436,-15,10.309113,0.237685,0.044359,1
2,2003,1113,1272,3,-1.896552,-0.172414,-0.024137,1
3,2003,1141,1166,5,-8.805643,-0.085684,-0.020792,1
4,2003,1143,1301,-1,0.324138,0.124138,-0.007780,1


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target
0,2010,3124,3201,-9,2.371212,-0.099432,0.053358,1
1,2010,3173,3395,-1,-0.730769,0.074359,-0.005113,1
2,2010,3181,3214,-13,8.831250,0.210417,0.054759,1
3,2010,3199,3256,-11,4.431183,0.091398,0.039516,1
4,2010,3207,3265,-7,-0.606061,0.012121,-0.087561,1


In [25]:
BASE_FEATURES = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff"
]


In [26]:
TARGET = "Target"
print(men[BASE_FEATURES + [TARGET]].head())

   SeedDiff  AvgScoringMarginDiff  WinRateDiff  EFGDifferentialDiff  Target
0         0              9.208046     0.151724             0.045943       0
1       -15             10.309113     0.237685             0.044359       1
2         3             -1.896552    -0.172414            -0.024137       1
3         5             -8.805643    -0.085684            -0.020792       1
4        -1              0.324138     0.124138            -0.007780       1


In [27]:
VALIDATION_SEASONS = [2017, 2018, 2019, 2021]


def temporal_cv_logistic(
    df,
    features,
    validation_seasons=VALIDATION_SEASONS,
    C=1.0,
    penalty=None
):
    
    results = []

    for val_season in validation_seasons:

        train = df[df["Season"] < val_season]
        validation = df[df["Season"] == val_season]

        X_train = train[features]
        y_train = train["Target"]

        X_val = validation[features]
        y_val = validation["Target"]

        logreg_kwargs = {
            "C": C,
            "solver": "lbfgs",
            "max_iter": 1000
        }
        if penalty is not None:
            logreg_kwargs["penalty"] = penalty

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(**logreg_kwargs))
        ])

        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_val)[:, 1]

        brier = brier_score_loss(
            y_val,
            probabilities
        )

        results.append({
            "ValidationSeason": val_season,
            "BrierScore": brier
        })

    results_df = pd.DataFrame(results)

    return results_df


In [28]:
men_results = temporal_cv_logistic(
    men,
    BASE_FEATURES
)

display(men_results)

,ValidationSeason,BrierScore
0,2017,0.173332
1,2018,0.217507
2,2019,0.176395
3,2021,0.220008


In [29]:
men_mean_brier = men_results["BrierScore"].mean()

print("Men mean CV Brier:", men_mean_brier)

Men mean CV Brier: 0.19681075204968873


In [30]:
women_results = temporal_cv_logistic(
    women,
    BASE_FEATURES
)

display(women_results)

women_mean_brier = women_results["BrierScore"].mean()

print("Women mean CV Brier:", women_mean_brier)

,ValidationSeason,BrierScore
0,2017,0.149460
1,2018,0.173040
2,2019,0.126360
3,2021,0.143551


Women mean CV Brier: 0.14810275315207297


In [31]:
C_VALUES = [
    0.001,
    0.01,
    0.1,
    0.5,
    1,
    2,
    5,
    10,
    100
]

In [32]:
tuning_results = []

for C in C_VALUES:

    results = temporal_cv_logistic(
        men,
        BASE_FEATURES,
        C=C
    )

    mean_brier = results["BrierScore"].mean()

    tuning_results.append({
        "C": C,
        "MeanBrier": mean_brier
    })

tuning_results = pd.DataFrame(tuning_results)

display(tuning_results.sort_values("MeanBrier"))

,C,MeanBrier
2,0.100,0.196329
3,0.500,0.196711
4,1.000,0.196811
5,2.000,0.196868
6,5.000,0.196909
7,10.000,0.196922
8,100.000,0.196934
1,0.010,0.198667
0,0.001,0.215769


In [33]:
women_tuning_results = []

for C in C_VALUES:

    results = temporal_cv_logistic(
        women,
        BASE_FEATURES,
        C=C
    )

    mean_brier = results["BrierScore"].mean()

    women_tuning_results.append({
        "C": C,
        "MeanBrier": mean_brier
    })

women_tuning_results = pd.DataFrame(women_tuning_results)

display(
    women_tuning_results.sort_values("MeanBrier")
)

,C,MeanBrier
2,0.100,0.146992
3,0.500,0.147369
4,1.000,0.148103
5,2.000,0.148668
6,5.000,0.149103
7,10.000,0.149269
8,100.000,0.149427
1,0.010,0.164077
0,0.001,0.212692


In [34]:
men_massey = pd.read_csv(
    DATA_DIR / "men_massey.csv"
)

display(men_massey.head())


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target,Team1MasseyRank,Team2MasseyRank,MasseyRankDiff
0,2003,1411,1421,0,9.208046,0.151724,0.045943,0,250.4,251.6,-1.2
1,2003,1112,1436,-15,10.309113,0.237685,0.044359,1,2.6,156.8,-154.2
2,2003,1113,1272,3,-1.896552,-0.172414,-0.024137,1,34.8,20.8,14.0
3,2003,1141,1166,5,-8.805643,-0.085684,-0.020792,1,40.8,23.0,17.8
4,2003,1143,1301,-1,0.324138,0.124138,-0.007780,1,33.8,47.0,-13.2


In [35]:
MASSEY_FEATURES = [
    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff",
    "MasseyRankDiff"
]

In [36]:
men_massey_results = temporal_cv_logistic(
    men_massey,
    MASSEY_FEATURES,
    C=1.0
)

display(men_massey_results)

print(
    "Mean Brier:",
    men_massey_results["BrierScore"].mean()
)

,ValidationSeason,BrierScore
0,2017,0.172475
1,2018,0.208809
2,2019,0.169063
3,2021,0.220218


Mean Brier: 0.19264099798694767


In [37]:
# STAGE 1: BASELINE RESULTS

print("MEN: Original four features")
display(tuning_results.sort_values("MeanBrier"))

best_men_baseline = tuning_results.sort_values(
    "MeanBrier"
).iloc[0]

print("Best men's C:", best_men_baseline["C"])
print("Best men's mean CV Brier:", best_men_baseline["MeanBrier"])


print("\nWOMEN: Original four features")
display(women_tuning_results.sort_values("MeanBrier"))

best_women_baseline = women_tuning_results.sort_values(
    "MeanBrier"
).iloc[0]

print("Best women's C:", best_women_baseline["C"])
print("Best women's mean CV Brier:", best_women_baseline["MeanBrier"])

MEN: Original four features


,C,MeanBrier
2,0.100,0.196329
3,0.500,0.196711
4,1.000,0.196811
5,2.000,0.196868
6,5.000,0.196909
7,10.000,0.196922
8,100.000,0.196934
1,0.010,0.198667
0,0.001,0.215769


Best men's C: 0.1
Best men's mean CV Brier: 0.1963293647291315

WOMEN: Original four features


,C,MeanBrier
2,0.100,0.146992
3,0.500,0.147369
4,1.000,0.148103
5,2.000,0.148668
6,5.000,0.149103
7,10.000,0.149269
8,100.000,0.149427
1,0.010,0.164077
0,0.001,0.212692


Best women's C: 0.1
Best women's mean CV Brier: 0.14699221951054664


In [38]:
# Individual validation-season scores for Stage 1

men_baseline_results = temporal_cv_logistic(
    men,
    BASE_FEATURES,
    C=float(best_men_baseline["C"])
)

women_baseline_results = temporal_cv_logistic(
    women,
    BASE_FEATURES,
    C=float(best_women_baseline["C"])
)

print("Men baseline scores:")
display(men_baseline_results)

print("Women baseline scores:")
display(women_baseline_results)

Men baseline scores:


,ValidationSeason,BrierScore
0,2017,0.176353
1,2018,0.213238
2,2019,0.176268
3,2021,0.219458


Women baseline scores:


,ValidationSeason,BrierScore
0,2017,0.148635
1,2018,0.164684
2,2019,0.129923
3,2021,0.144726


In [39]:
# STAGE 2: MERGE ELO INTO MATCHUP DATA

men_elo = pd.read_csv("../data/processed/men_elo.csv")
women_elo = pd.read_csv("../data/processed/women_elo.csv")


def merge_elo(matchups, elo):
    # Rename TeamID to match each side of the matchup
    team1_elo = elo.rename(columns={
        "TeamID": "Team1",
        "FinalElo": "Team1Elo"
    })

    team2_elo = elo.rename(columns={
        "TeamID": "Team2",
        "FinalElo": "Team2Elo"
    })

    # Merge Team1 Elo
    merged = matchups.merge(
        team1_elo[["Season", "Team1", "Team1Elo"]],
        on=["Season", "Team1"],
        how="left",
        validate="many_to_one"
    )

    # Merge Team2 Elo
    merged = merged.merge(
        team2_elo[["Season", "Team2", "Team2Elo"]],
        on=["Season", "Team2"],
        how="left",
        validate="many_to_one"
    )

    # Calculate Elo difference
    merged["EloDiff"] = (
        merged["Team1Elo"] - merged["Team2Elo"]
    )

    # Verify successful merging
    missing = merged[
        ["Team1Elo", "Team2Elo", "EloDiff"]
    ].isna().sum()

    print("Missing Elo values:")
    print(missing)

    assert merged[
        ["Team1Elo", "Team2Elo"]
    ].notna().all().all(), (
        "Some teams are missing Elo ratings. "
        "Check the source data before training."
    )

    return merged


men_elo_merged = merge_elo(men, men_elo)
women_elo_merged = merge_elo(women, women_elo)

print("Men merged shape:", men_elo_merged.shape)
print("Women merged shape:", women_elo_merged.shape)

display(
    men_elo_merged[
        ["Season", "Team1", "Team2", "EloDiff", "Target"]
    ].head()
)

Missing Elo values:
Team1Elo    0
Team2Elo    0
EloDiff     0
dtype: int64


Missing Elo values:
Team1Elo    0
Team2Elo    0
EloDiff     0
dtype: int64
Men merged shape: (1449, 11)
Women merged shape: (961, 11)


,Season,Team1,Team2,EloDiff,Target
0,2003,1411,1421,5.667214,0
1,2003,1112,1436,190.885235,1
2,2003,1113,1272,-150.632708,1
3,2003,1141,1166,-22.348118,1
4,2003,1143,1301,-2.834551,1


In [41]:
# Helper function: tune C for a given dataset and feature set

C_VALUES = [
    0.001,
    0.01,
    0.1,
    0.5,
    1,
    2,
    5,
    10,
    100
]

def tune_C(df, features):
    rows = []

    for C in C_VALUES:
        scores = temporal_cv_logistic(
            df,
            features,
            C=C
        )

        rows.append({
            "C": C,
            "MeanBrier": scores["BrierScore"].mean(),
            **{
                f"Brier_{int(row.ValidationSeason)}":
                    row.BrierScore
                for row in scores.itertuples()
            }
        })

    return pd.DataFrame(rows).sort_values("MeanBrier").reset_index(drop=True)

In [42]:
# STAGE 2: ORIGINAL FOUR + ELO

ELO_FEATURES = BASE_FEATURES + ["EloDiff"]

men_elo_results = tune_C(
    men_elo_merged,
    ELO_FEATURES
)

women_elo_results = tune_C(
    women_elo_merged,
    ELO_FEATURES
)

print("Men: Four features + Elo")
display(men_elo_results)

print("Women: Four features + Elo")
display(women_elo_results)

Men: Four features + Elo


,C,MeanBrier,Brier_2017,Brier_2018,Brier_2019,Brier_2021
0,0.100,0.197306,0.184805,0.210413,0.172150,0.221857
1,0.500,0.198008,0.184881,0.213508,0.171309,0.222333
2,1.000,0.198219,0.185141,0.214105,0.171210,0.222421
3,2.000,0.198357,0.185312,0.214433,0.171163,0.222518
4,5.000,0.198440,0.185431,0.214641,0.171136,0.222553
5,10.000,0.198470,0.185474,0.214712,0.171127,0.222566
6,100.000,0.198498,0.185514,0.214782,0.171119,0.222577
7,0.010,0.199909,0.195037,0.206629,0.178026,0.219945
8,0.001,0.214142,0.214816,0.219188,0.201055,0.221510


Women: Four features + Elo


,C,MeanBrier,Brier_2017,Brier_2018,Brier_2019,Brier_2021
0,0.500,0.145996,0.146474,0.167311,0.120264,0.149935
1,0.100,0.146273,0.147779,0.162017,0.125986,0.149309
2,1.000,0.146477,0.146835,0.168560,0.119102,0.151409
3,2.000,0.146875,0.147178,0.169267,0.118468,0.152586
4,5.000,0.147212,0.147493,0.169767,0.118113,0.153477
5,10.000,0.147338,0.147626,0.169938,0.117977,0.153812
6,100.000,0.147442,0.147736,0.170089,0.117852,0.154092
7,0.010,0.163595,0.167623,0.166194,0.150199,0.170363
8,0.001,0.208743,0.215460,0.207825,0.202512,0.209174


In [43]:
best_men_elo = men_elo_results.iloc[0]
best_women_elo = women_elo_results.iloc[0]

print("Men best C:", best_men_elo["C"])
print("Men mean Brier:", best_men_elo["MeanBrier"])

print("Women best C:", best_women_elo["C"])
print("Women mean Brier:", best_women_elo["MeanBrier"])

Men best C: 0.1
Men mean Brier: 0.19730637278217558
Women best C: 0.5
Women mean Brier: 0.14599602439257164


In [44]:
# STAGE 3: MERGE ELO INTO MEN'S MASSEY DATA

men_massey_elo = merge_elo(
    men_massey,
    men_elo
)

MASSEY_ELO_FEATURES = BASE_FEATURES + [
    "MasseyRankDiff",
    "EloDiff"
]

men_massey_elo_results = tune_C(
    men_massey_elo,
    MASSEY_ELO_FEATURES
)

display(men_massey_elo_results)

best_men_massey_elo = men_massey_elo_results.iloc[0]

print("Best C:", best_men_massey_elo["C"])
print("Mean Brier:", best_men_massey_elo["MeanBrier"])

Missing Elo values:
Team1Elo    0
Team2Elo    0
EloDiff     0
dtype: int64


,C,MeanBrier,Brier_2017,Brier_2018,Brier_2019,Brier_2021
0,0.100,0.193696,0.183086,0.203393,0.166442,0.221864
1,0.500,0.194244,0.184174,0.205300,0.165223,0.222277
2,1.000,0.194413,0.184583,0.205646,0.165051,0.222372
3,2.000,0.194510,0.184833,0.205834,0.164962,0.222412
4,5.000,0.194575,0.185001,0.205951,0.164909,0.222437
5,10.000,0.194597,0.185060,0.205991,0.164891,0.222446
6,100.000,0.194618,0.185114,0.206028,0.164875,0.222454
7,0.010,0.195279,0.189731,0.200479,0.172218,0.218690
8,0.001,0.208840,0.208582,0.213516,0.195094,0.218167


Best C: 0.1
Mean Brier: 0.19369621222730415


In [45]:
# STAGE 4: ORIGINAL FOUR + MASSEY

MASSEY_FEATURES = BASE_FEATURES + ["MasseyRankDiff"]

men_massey_results = tune_C(
    men_massey,
    MASSEY_FEATURES
)

display(men_massey_results)

best_men_massey = men_massey_results.iloc[0]

print("Best C:", best_men_massey["C"])
print("Mean Brier:", best_men_massey["MeanBrier"])

,C,MeanBrier,Brier_2017,Brier_2018,Brier_2019,Brier_2021
0,0.100,0.192534,0.174697,0.205991,0.169629,0.219820
1,0.500,0.192614,0.172742,0.208396,0.169138,0.220179
2,1.000,0.192641,0.172475,0.208809,0.169063,0.220218
3,2.000,0.192652,0.172341,0.209017,0.169013,0.220238
4,10.000,0.192660,0.172217,0.209191,0.168979,0.220256
5,5.000,0.192662,0.172261,0.209147,0.168988,0.220251
6,100.000,0.192663,0.172192,0.209230,0.168971,0.220260
7,0.010,0.194099,0.184749,0.201656,0.174035,0.215956
8,0.001,0.209803,0.208073,0.215422,0.198445,0.217273


Best C: 0.1
Mean Brier: 0.19253447463173


In [46]:
# FINAL COMPARISON OF THE FOUR STAGES

stage_results = [
    {
        "Gender": "Men",
        "Stage": "1. Baseline",
        "BestC": best_men_baseline["C"],
        "MeanBrier": best_men_baseline["MeanBrier"]
    },
    {
        "Gender": "Men",
        "Stage": "2. Baseline + Elo",
        "BestC": best_men_elo["C"],
        "MeanBrier": best_men_elo["MeanBrier"]
    },
    {
        "Gender": "Men",
        "Stage": "3. Baseline + Massey + Elo",
        "BestC": best_men_massey_elo["C"],
        "MeanBrier": best_men_massey_elo["MeanBrier"]
    },
    {
        "Gender": "Men",
        "Stage": "4. Baseline + Massey",
        "BestC": best_men_massey["C"],
        "MeanBrier": best_men_massey["MeanBrier"]
    },
    {
        "Gender": "Women",
        "Stage": "1. Baseline",
        "BestC": best_women_baseline["C"],
        "MeanBrier": best_women_baseline["MeanBrier"]
    },
    {
        "Gender": "Women",
        "Stage": "2. Baseline + Elo",
        "BestC": best_women_elo["C"],
        "MeanBrier": best_women_elo["MeanBrier"]
    }
]

stage_comparison = pd.DataFrame(stage_results)

display(
    stage_comparison.sort_values(
        ["Gender", "MeanBrier"]
    )
)

,Gender,Stage,BestC,MeanBrier
3,Men,4. Baseline + Massey,0.1,0.192534
2,Men,3. Baseline + Massey + Elo,0.1,0.193696
0,Men,1. Baseline,0.1,0.196329
1,Men,2. Baseline + Elo,0.1,0.197306
5,Women,2. Baseline + Elo,0.5,0.145996
4,Women,1. Baseline,0.1,0.146992


In [48]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def make_logistic_model(C=1.0):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            C=C,
            penalty="l2",
            solver="lbfgs",
            max_iter=3000
        ))
    ])

In [49]:
HOLDOUT_SEASONS = [2022, 2023, 2024, 2025]

# Best configurations from your CV comparison
BEST_MEN_FEATURES = BASE_FEATURES + ["MasseyRankDiff"]
BEST_WOMEN_FEATURES = BASE_FEATURES + ["EloDiff"]

# Fit using all historical data through 2021
men_train = men_massey_elo[
    men_massey_elo["Season"] <= 2021
]
women_train = women_elo_merged[
    women_elo_merged["Season"] <= 2021
]

men_model = make_logistic_model(C=0.1)
women_model = make_logistic_model(C=0.5)

men_model.fit(
    men_train[BEST_MEN_FEATURES],
    men_train["Target"]
)

women_model.fit(
    women_train[BEST_WOMEN_FEATURES],
    women_train["Target"]
)


def evaluate_later_seasons(
    df,
    model,
    features,
    seasons=HOLDOUT_SEASONS,
):
    results = []

    for season in seasons:
        test = df[df["Season"] == season]

        if test.empty:
            print(f"Warning: no data for {season}")
            continue

        probabilities = model.predict_proba(
            test[features]
        )[:, 1]

        results.append({
            "Season": season,
            "N": len(test),
            "BrierScore": brier_score_loss(
                test["Target"],
                probabilities,
            ),
        })

    return pd.DataFrame(results)


men_holdout = evaluate_later_seasons(
    men_massey_elo,
    men_model,
    BEST_MEN_FEATURES,
)

women_holdout = evaluate_later_seasons(
    women_elo_merged,
    women_model,
    BEST_WOMEN_FEATURES,
)

print("MEN: 2022–2025")
display(men_holdout)
if not men_holdout.empty:
    print("Mean holdout Brier:", men_holdout["BrierScore"].mean())

print("WOMEN: 2022–2025")
display(women_holdout)
if not women_holdout.empty:
    print("Mean holdout Brier:", women_holdout["BrierScore"].mean())

MEN: 2022–2025


c:\Users\katri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\katri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,Season,N,BrierScore
0,2022,67,0.230594
1,2023,67,0.209540
2,2024,67,0.199773
3,2025,67,0.157488


Mean holdout Brier: 0.19934887498394965
WOMEN: 2022–2025


,Season,N,BrierScore
0,2022,67,0.171905
1,2023,67,0.163917
2,2024,67,0.120957
3,2025,67,0.115488


Mean holdout Brier: 0.14306675685704778


In [50]:
# Baseline models: original four features only

men_baseline_train = men[
    men["Season"] <= 2021
].copy()

women_baseline_train = women[
    women["Season"] <= 2021
].copy()

men_baseline_model = make_logistic_model(C=0.1)
women_baseline_model = make_logistic_model(C=0.1)

men_baseline_model.fit(
    men_baseline_train[BASE_FEATURES],
    men_baseline_train["Target"]
)

women_baseline_model.fit(
    women_baseline_train[BASE_FEATURES],
    women_baseline_train["Target"]
)

men_baseline_holdout = evaluate_later_seasons(
    men,
    men_baseline_model,
    BASE_FEATURES
)

women_baseline_holdout = evaluate_later_seasons(
    women,
    women_baseline_model,
    BASE_FEATURES
)

print("MEN: Baseline holdout")
display(men_baseline_holdout)

print("WOMEN: Baseline holdout")
display(women_baseline_holdout)

MEN: Baseline holdout


c:\Users\katri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\katri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,Season,N,BrierScore
0,2022,67,0.233230
1,2023,67,0.206762
2,2024,67,0.200159
3,2025,67,0.159661


WOMEN: Baseline holdout


,Season,N,BrierScore
0,2022,67,0.163390
1,2023,67,0.164860
2,2024,67,0.118511
3,2025,67,0.122555
